In [ ]:
from collections import Counter, deque
from functools import lru_cache
import re
import json

class BPETokenizerSimple:
    def __init__(self):
        self.vocab= {}  #maps token ids to token_str
        self.inverse_vocab= {} #maps token str to toke ids
        self.bpe_merges ={} #dictionary of bpe merges
        self.bpe_ranks ={} 

#           training text  desired vocab size    set of special tokens to be included
    def train(self, text, vocab_size, allowed_specials={"<|endoftext|>"}):
        tokens =self.pretokenize_text(text)
        unique_chars =[chr(i) for i in range(256)]
        unique_chars.extend(
            char for char in sorted({char for token in tokens for char in token})
            if char not in unique_chars
        )
        if "Ġ" not in unique_chars:
            unique_chars.append("Ġ")

        self.vocab= {i: char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char: i for i, char in self.vocab.items()}

        # addin allowed special tokens
        if allowed_specials:
            for token in allowed_specials:
                if token not in self.inverse_vocab:
                    new_id =len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] =new_id
        #tokenizin each pretoken into char ids
        token_id_sequences= [
            [self.inverse_vocab[char] for char in token]
            for token in tokens
        ]
        # replacin and indin frequent pairs
        for new_id in range(len(self.vocab), vocab_size):
            pair_id =self.find_freq_pair(token_id_sequences, mode="most")
            if pair_id is None:
                break
            token_id_sequences = self.replace_pair(token_id_sequences, pair_id, new_id)
            self.bpe_merges[pair_id] =new_id
        for(p0, p1), new_id in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id

    # loading pretrained vocab and bpe merges 
    def load_vocab_and_merges_from_openai(self, vocab_path, bpe_merges_path):
        with open(vocab_path, "r", encoding="utf-8") as file:
            loaded_vocab =json.load(file)
            self.vocab = {int(v): k for k, v in loaded_vocab.items()}
            self.inverse_vocab = {k: int(v) for k, v in loaded_vocab.items()}

        if "Ċ" not in self.inverse_vocab or self.inverse_vocab["Ċ"] !=198:
            raise KeyError("Vocabulary missing gpt-2 newlien glyph 'Ċ' at id 198.")
        # must have endoftext at 50256
        if "<|endoftext|>" not in self.inverse_vocab or self.inverse_vocab["<|endoftext|>"] != 50256:
            raise KeyError("Vocabulary missing <|endoftext|> at id 50256.")
        
        if "\n" not in self.inverse_vocab:
            self.inverse_vocab["\n"]= self.inverse_vocab["C"]
        if "\r" not in self.inverse_vocab:
            if 201 in self.vocab:
                self.inverse_vocab["\r"]= 201
            else:
                raise KeyError("Vocabulary missing carriage return token at id 201")

        # loading gpt2 merges and stores ranks
        self.bpe_ranks ={}
        with open(bpe_merges_path, "r", encoding="utf-8") as file:
            lines= file.readlines()
            if lines and lines[0].startswith("#"):
                lines = lines[1:]
            rank= 0
            for line in lines:
                t1, *rest= line.strip().split()
                if len(rest)!= 1:
                    continue
                t2 = rest[0]
                if t1 in self.inverse_vocab and t2 in self.inverse_vocab:
                    self.bpe_ranks[(t1, t2)] = rank
                    rank += 1
                else:
                    pass
    # encoding input text into list of token ids
    def encode(self, text, allowed_special=None):
        specials_in_vocab= [
            tok for tok in self.inverse_vocab:
            if tok.startswith("<|") and tok.endswith("|>")
        ]
        if allowed_special is None:
            disallowed =[tok for tok in self.inverse_vocab if tok in text]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")
            else:
                disallowed= [tok for tok in specials_in_vocab if tok in text and tok not in allowed_special]
                if disallowed:
                    raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")

        token_ids =[]
        if allowed_special is not None and len(allowed_special) >0:
            special_pattern= "(" + "|".join(re.escape(tok) for tok in sorted(allowed_special, key=len, reverse=True)) + ")"
            last_index =0

            for match in re.finditer(special_pattern, text):
                prefix =text[last_index:match.start()]
                token_ids.extend(self.encode(prefix, allowed_special=None))
                special_token= match.group(0)
                if special_token in self.inverse_vocab:
                    token_ids.append(self.inverse_vocab[special_token])
                else:
                    raise ValueError(f"Special token {special_token} not found in vocabulary.")
                last_index= match.end()

            text= text[last_index:]
            disallowed = [
                tok for tok in self.inverse_vocab
                if tok.startswith("<|") and tok.endswith("|>") and tok in text and tok not in allowed_special
            ]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")
            
        tokens =self.pretokenize_text(text)
        for tok in tokens:  #mappin tokens to ids
            if tok in self.inverse_vocab:
                token_ids.append(self.inverse_vocab[tok])
            else:
                token_ids.extend(self.tokenize_with_bpe(tok))
        return token_ids

    @lru_cache(maxsize=None)
    def get_special_token_id(self, token):
        return self.inverse_vocab.get(token, None)

    @staticmethod
    def pretokenize_text(text):
        tokens =[]
        parts = re.split(r'(\r\n|\r|\n)', text)

        for part in parts:
            if part== "":
                continue
            if part =="\r\n":
                tokens.append("\r")
                tokens.append("\n")
                continue
            if part == "\r":
                tokens.append("\r")
                continue
            if part == "\n":
                tokens.append("\n")
                continue

            pending_spaces= 0
            for m in re.finditer(r'( +)|(\S+)', part):
                if m.group(1) is not None:
                    pending_spaces +=len(m.group(1))
                else:
                    word= m.group()
                    if pending_spaces> 0:
                        for _ in range(pending_spaces -1):
                            tokens.append("Ġ")  # remaining spaces as standalone
                        tokens.append("Ġ" +word) #one leading space
                        pending_spaces =0
                    
                    else:
                        tokens.append(word)
            for _ in range(pending_spaces):
                tokens.append("Ġ")
        return tokens
    
    @staticmethod
    def find_freq_pair(token_id_sequences, mode="most"):
        pairs = Counter(
            pair
            for token_ids in token_id_sequences
            for pair in zip(token_ids, token_ids[1:])
        )
        if not pairs: 
            return None
        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalid mode. Choose 'most' or 'least'.")

    @staticmethod
    def replace_pair(token_id_sequences, pair_id, new_id):
        replace_sequences= []
        for token_ids in token_id_sequences:
            dq= deque(token_ids)
            replaced= []
            while dq:
                curr = dq.popleft()
                if dq and (curr, dq[0])== pair_id:
                    replaced.append(new_id)
                    dq.popleft()
                else:
                    replaced.append(curr)
            
            replace_sequences.append(replaced)
        return replace_sequences